# Regression for house prices

Final model Kaggle RMSE: 27,003.58788.

This notebook is the code for our final model, followed by a record of our tests. Detailed improving process are recorded in README.md.

## 1.Load and inspect the data

We first load the training data, test data, and sample submission file. The training dataset contains 1,200 houses with known sale prices, while the test dataset contains 260 houses whose prices we need to predict.

We use SalePrice as our target variable. The other housing features, such as overall quality, living area, and neighborhood, will be used to predict it.

Before building the model, we check the following:

1. Dataset size: We use .shape to confirm the number of rows and columns in the training and test datasets.
2. Missing values: We count missing values in the training predictors to see whether we need to handle incomplete data.
3. Sample observations: We display the first few rows of Id, OverallQual, GrLivArea, and SalePrice to get an initial understanding of the data.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
train = pd.read_csv(ROOT/"data"/"train.csv")
test = pd.read_csv(ROOT/"data"/"test.csv")
sample = pd.read_csv(ROOT/"data"/"sample_soln.csv")

# Check dataset sizes and count missing values
print("Train:", train.shape, "Test:", test.shape)
print("Missing cells in training predictors:", train.drop(columns="SalePrice").isna().sum().sum())

# Preview important housing features and their sale prices
display(train[["Id", "OverallQual", "GrLivArea", "SalePrice"]].head())

# Check that the test data and submission format are valid
assert len(test)==len(sample)== 260 and "SalePrice" not in test.columns
assert sample["Id"].is_unique

Train: (1200, 81) Test: (260, 80)
Missing cells in training predictors: 6441


,Id,OverallQual,GrLivArea,SalePrice
0,1,7,1710,208500
1,2,6,1262,181500
2,3,7,1786,223500
3,4,7,1717,140000
4,5,8,2198,250000


## 2. Inspect unusual large homes and prepare features

We created several new features to help the model capture relationships between house characteristics and sale prices. We also capped living area at 4,000 square feet to reduce extreme predictions and added interactions between overall quality, living area, and garage spaces. Detailed features selection process are recorded in the 6. Model refining process part at the end of the code.

In [2]:
# Find houses with more than 4,000 square feet of living area
large_train = train.loc[train["GrLivArea"]>4000,["Id", "Neighborhood", "GrLivArea", "SalePrice"]]
large_test = test.loc[test["GrLivArea"]>4000,["Id", "Neighborhood", "GrLivArea"]]

# Compare large houses in the training and test datasets
display(large_train)
display(large_test)

# Count large houses in Edwards in both datasets
print("Matching train rows:", ((train.Neighborhood == "Edwards")& (train.GrLivArea > 4000)).sum())
print("Matching test rows:", ((test.Neighborhood == "Edwards") &(test.GrLivArea > 4000)).sum())

,Id,Neighborhood,GrLivArea,SalePrice
523,524,Edwards,4676,184750
691,692,NoRidge,4316,755000
1182,1183,NoRidge,4476,745000


,Id,Neighborhood,GrLivArea
98,1299,Edwards,5642


Matching train rows: 1
Matching test rows: 1


In [3]:
# Create additional features from the original housing data
def make_features(frame, area_cap=None, large_edwards=False, garage_interaction=False):

    # Remove house IDs and the target variable from predictors
    x = frame.drop(columns=["Id", "SalePrice"], errors="ignore").copy()
    # Cap living area to reduce extreme predictions for very large houses
    if area_cap is not None:
        x["GrLivArea"]=x["GrLivArea"].clip(upper=area_cap)

    # Combine basement, first-floor, and second-floor areas
    x["TotalArea"]=x[["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]].sum(axis=1, min_count=3)

    # Calculate total bathrooms, counting half bathrooms as 0.5
    x["TotalBathrooms"]=(x["FullBath"]+0.5*x["HalfBath"]+x["BsmtFullBath"] + 0.5*x["BsmtHalfBath"])

    # Calculate the age of each house when it was sold
    x["HouseAge"] = x["YrSold"] - x["YearBuilt"]

    # Allow living area to have a different effect at different quality levels
    x["QualityLivingArea"] = x["OverallQual"] * x["GrLivArea"]

    # Treat house type codes as categories rather than numerical values
    x["MSSubClass"] = x["MSSubClass"].astype(str)
    # Identify unusually large houses in Edwards for a separate price adjustment
    if large_edwards:
        x["LargeEdwards"]=((frame["Neighborhood"] == "Edwards")&(frame["GrLivArea"] > 4000)).astype(int)
    # Allow the value of garage spaces to depend on overall house quality
    if garage_interaction:
        x["QualityGarageCars"]=x["OverallQual"]* x["GarageCars"]
    return x

# Build a pipeline to handle missing values, encoding, and scaling
def make_pipeline(x, model):
    # Separate numerical and categorical predictors
    numeric = x.select_dtypes(include="number").columns.tolist()
    categorical = x.select_dtypes(exclude="number").columns.tolist()
    # Fill numerical missing values with medians and standardize the features
    preparation = ColumnTransformer([("numeric", Pipeline([("fill", SimpleImputer(strategy="median")),("scale", StandardScaler())]), numeric),
        ("categorical", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="Missing")),("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)])
    # Combine preprocessing and regression into one pipeline
    return Pipeline([("prepare", preparation), ("model", model)])

# Set SalePrice as the target and check the number of final predictors
y = train["SalePrice"]
print("Final predictors:", make_features(train, 4000, True, True).shape[1])


Final predictors: 85


## 3. Compare versions with repeated cross-validation

We compared three Ridge models using five-fold cross-validation with five different random seeds. We calculated RMSE for each run and compared the average scores to see which combination of features performed best.

In [4]:
# Store validation results for each model
rows = []
# Compare three Ridge models with different engineered features
versions = [
    ("Capped Ridge", False, False),
    ("Ridge + LargeEdwards", True, False),
    ("Final Ridge + garage interaction", True, True)]

# Repeat five-fold cross-validation using five different random seeds
for seed in [42, 2026, 2027, 2028, 17]:
    # Use the same folds to make a fair comparison between models
    folds = KFold(n_splits=5, shuffle=True, random_state=seed)
    for name, large_edwards, garage_interaction in versions:
        # Prepare features and build a Ridge model for each version
        x = make_features(train, area_cap=4000,
                          large_edwards=large_edwards,
                          garage_interaction=garage_interaction)
        pipe = make_pipeline(x, Ridge(alpha=10, solver="lsqr"))

        # Generate out-of-fold predictions to evaluate each model
        predictions = cross_val_predict(pipe, x, y, cv=folds, n_jobs=4)
        # Calculate RMSE and save the result for each model
        rows.append({"seed": seed, "model": name,
                     "oof_rmse": np.sqrt(mean_squared_error(y, predictions))})

# Display RMSE results for each random seed and model
results = pd.DataFrame(rows)
display(results.pivot(index="seed", columns="model", values="oof_rmse").round(2))

# Calculate average RMSE and rank models from best to worst
mean_scores = results.groupby("model")["oof_rmse"].mean().sort_values()
display(mean_scores.round(2))


model,Capped Ridge,Final Ridge + garage interaction,Ridge + LargeEdwards
seed,,,
17,27889.52,27112.62,27283.00
42,28407.39,27368.73,27494.52
2026,27974.51,26952.64,27117.22
2027,28600.85,27404.66,27585.03
2028,28093.63,27081.41,27255.94


model
Final Ridge + garage interaction    27184.01
Ridge + LargeEdwards                27347.14
Capped Ridge                        28193.18
Name: oof_rmse, dtype: float64

## 4. Fit the final model

We selected the Ridge model with the lowest average validation RMSE and trained it on all 1,200 training houses. We then predicted prices for the 260 test houses and saved the results in the format required by Kaggle.

In [5]:
# Confirm that the final Ridge model has the lowest validation RMSE
assert mean_scores.index[0] == "Final Ridge + garage interaction"

# Apply the same feature engineering to training and test data
x_train = make_features(train, 4000, True, True)
x_test = make_features(test, 4000, True, True)

# Train the final Ridge model using all 1,200 training houses
final_model = make_pipeline(x_train, Ridge(alpha=10, solver="lsqr"))
final_model.fit(x_train, y)
# Predict test house prices and prevent negative predictions
predicted_prices = np.maximum(final_model.predict(x_test), 0)
# Create the Kaggle submission with house IDs and predicted prices
submission = pd.DataFrame({"Id": sample["Id"], "SalePrice": predicted_prices})

# Check the submission size, ID order, and predicted values
assert submission.shape==(260, 2)
assert submission["Id"].equals(sample["Id"])
assert submission["Id"].is_unique
assert np.isfinite(submission["SalePrice"]).all()

# Save predictions as a CSV file for Kaggle
output = ROOT/"submission.csv"
submission.to_csv(output, index=False)

# Display the final validation RMSE and preview the predictions
print("Saved:", output)
print("Five-partition mean validation RMSE:", round(mean_scores.iloc[0], 2))
display(submission.head())

Saved: /Users/zyh/Desktop/submission.csv
Five-partition mean validation RMSE: 27184.01


,Id,SalePrice
0,0,94093.469079
1,1,198364.090443
2,2,134159.294635
3,3,198508.140423
4,4,142202.699370


## 5. Final method

Our final model used Ridge regression with degree-2 polynomial features to capture relationships between variables. We used six predictors: floors, living area, bathrooms, lot size, above-ground area, and house grade. Polynomial features created additional combinations of these variables. Ridge regression then used these features to predict house prices while limiting overly large coefficients.

LargeEdwards equals 1 for homes located in Edwards with more than 4,000 square feet of living space. This lets the model account for this specific group of homes separately. Since there are only a few examples, its impact should be interpreted carefully. QualityGarageCars measures the relationship between house quality and the number of garage spaces.

Across five repeated five-fold partitions, the final model's mean RMSE is 27,184.01, versus 27,347.14 for the previous Ridge with LargeEdwards. Its actual public Kaggle RMSE is 27,003.58788, lower than the previous 27,643.51679. The final private 40% of hidden labels can score differently. Public feedback guided model development, so neither the local nor public score guarantees future performance.

## 6. Models we submitted to Kaggle

detailed process are recorded in README.md

| Submission | Kaggle File Name | Model | Public RMSE |
| --- | --- | --- | ---: | 
| Simple random forest | submission_04.csv | RandomForestRegressor | 30,697.63354 |
| Capped Ridge | submission_1101-05.csv | Ridge without LargeEdwards | 54,417.66090 |
| Previous best Ridge | submission_1101-6.csv | Ridge with LargeEdwards | 27,643.51679 | 
| Ridge plus forest average | submission_1102-1.csv | 75% previous Ridge, 25% forest | 27,802.62697 |
| **Final submission** | submission_1102-2.csv | One Ridge with QualityGarageCars | **27,003.58788** |